In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

### cell 10 – further optimized for cuDF (fixed)

# 1) restrict to offense (1) and defense (0) only, matching the original code
los_team = (
    df
      .loc[df['posTeam'].isin([0, 1]), ['gameId','playId','nflId','posTeam','los_diff']]
      .groupby(['gameId','playId','nflId','posTeam'])
      .los_diff.agg(['max','min'])
      .reset_index()
)
# 2) pick max for offense (1), min for defense (0)
los = (
    los_team
      .assign(los_diff = los_team['max'].where(los_team['posTeam']==1,
                                               los_team['min']))
      [['nflId','los_diff']]
)
# 3) average LOS diff per player
top = los.groupby('nflId', as_index=False).los_diff.mean()
# 4) merge back onto snap_speed and rename
snap_speed = (
    snap_speed
      .merge(top, on='nflId', how='inner')
      .rename(columns={'s':'snap_s',
                       'a':'snap_a',
                       'los_diff':'snap_los_diff'})
)